# Heart Disease Prediction — Ensemble Notebook

**Competition:** Playground Series S6E2  
**Approach:** CatBoost (base + alt variants) × RealMLP × XGBoost → Meta Ridge blending

| | Score |
|---|---|
| **CV (OOF AUC)** | 0.955720 |
| **Public LB** | 0.95396 |

## 1. Environment & Installs


In [1]:
# Install required packages
!pip install pytabkit catboost -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 6.6 MB/s eta 0:00:00


## 2. Imports & Global Config


In [2]:
# Standard library
import json
import subprocess
import warnings
import zipfile
from pathlib import Path

# Numerical / data
import numpy as np
import pandas as pd

# Machine learning
import torch
from catboost import CatBoostClassifier, Pool
from pytabkit import RealMLP_TD_Classifier
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

# Global config
CFG = dict(
    comp_slug       = "playground-series-s6e2",
    target_col      = "Heart Disease",
    id_col          = "id",
    n_folds         = 5,
    random_state    = 42,
    # Paths
    kaggle_comp_dir = Path("/kaggle/input/competitions/playground-series-s6e2"),
    kaggle_ext_path = Path("/kaggle/input/datasets/neurocipher/heartdisease/Heart_Disease_Prediction.csv"),
    local_data_dir  = Path("data/raw"),
    need_files      = ["train.csv", "test.csv", "sample_submission.csv"],
)

# Set all random seeds
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(CFG['random_state'])
np.random.seed(CFG['random_state'])
if DEVICE == 'cuda':
    torch.cuda.manual_seed_all(CFG['random_state'])
    torch.set_float32_matmul_precision('high')

CFG['local_data_dir'].mkdir(parents=True, exist_ok=True)
print(f"Device: {DEVICE}")


Device: cuda


## 3. Utilities


In [3]:
# Kaggle / local data helpers
def _run(cmd: list[str]) -> None:
    """Run a subprocess command; raises on non-zero exit."""
    subprocess.run(cmd, capture_output=True, text=True, check=True)

def _ensure_kaggle_json_colab(dst: Path = Path("/content/kaggle.json")) -> Path:
    """Prompt an upload dialog in Colab if kaggle.json is missing; otherwise require it exists."""
    if dst.exists():
        print("Found:", dst)
        return dst
    try:
        from google.colab import files  # type: ignore
    except ImportError:
        raise FileNotFoundError(
            f"{dst} not found. Place kaggle.json at {dst} or set up credentials another way."
        )
    print("Upload your kaggle.json (Kaggle → Account → API → Create New Token)")
    uploaded = files.upload()
    cand = next((n for n in uploaded if n.endswith(".json")), None)
    if cand is None:
        raise FileNotFoundError("Upload failed: no .json file received.")
    Path(cand).rename(dst)
    print("Saved to:", dst)
    return dst

def _install_kaggle_json(src: Path) -> None:
    """Copy kaggle.json → ~/.kaggle/kaggle.json with chmod 600."""
    if not src.exists():
        raise FileNotFoundError(f"{src} not found.")
    dst_dir = Path.home() / ".kaggle"
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / "kaggle.json"
    dst.write_bytes(src.read_bytes())
    try:
        dst.chmod(0o600)
    except Exception:
        pass
    cfg = json.loads(dst.read_text())
    if "username" not in cfg or "key" not in cfg:
        raise ValueError("kaggle.json is missing 'username' or 'key'.")
    print(f"Installed kaggle.json for user: {cfg['username']}")

def _local_data_ready(data_dir: Path) -> bool:
    return all((data_dir / f).exists() for f in CFG['need_files'])

def _download_competition(data_dir: Path) -> None:
    """Download and unzip competition data via kaggle CLI."""
    _run(["kaggle", "config", "view"])
    _run(["kaggle", "competitions", "download", "-c", CFG['comp_slug'], "-p", str(data_dir), "--force"])
    zips = list(data_dir.glob("*.zip"))
    if not zips:
        raise FileNotFoundError(f"No zip found in {data_dir} after download.")
    for zp in zips:
        with zipfile.ZipFile(zp, "r") as zf:
            zf.extractall(data_dir)
        print("Unzipped:", zp.name)
    if not _local_data_ready(data_dir):
        missing = [f for f in CFG['need_files'] if not (data_dir / f).exists()]
        raise FileNotFoundError(f"Missing after download: {missing}")

# Preprocessing helpers
def encode_target_strict(y: pd.Series) -> pd.Series:
    """Map common string target labels to {0, 1}. Raises on unknown values."""
    mapping_candidates = [
        {"No": 0, "Yes": 1}, {"N": 0, "Y": 1},
        {"Negative": 0, "Positive": 1}, {"Absent": 0, "Present": 1},
        {"Absence": 0, "Presence": 1}, {0: 0, 1: 1}, {"0": 0, "1": 1},
    ]
    uniq = set(pd.Series(y).dropna().unique().tolist())
    for mp in mapping_candidates:
        if uniq.issubset(set(mp.keys())):
            return pd.Series(y).map(mp).astype("int8")
    raise ValueError(f"Unknown target labels: {sorted(list(uniq))}")

def check_data_quality(df: pd.DataFrame, name: str = "Dataset") -> None:
    """Print a concise data quality summary."""
    cols = [c for c in df.columns if c != 'id']
    dupes = df.duplicated(subset=cols).sum()
    nans  = df.isnull().sum().sum()
    print(f"[{name}] rows={len(df)}, duplicates={dupes}, total_NaN={nans}")
    if nans > 0:
        nan_series = df.isnull().sum()
        print(nan_series[nan_series > 0].to_string())

# Feature engineering helpers
def add_external_target_stats(
    df: pd.DataFrame,
    original_df: pd.DataFrame | None,
    base_features: list[str],
    target_col: str,
) -> pd.DataFrame:
    """Merge group-wise target statistics from the external/original dataset into df."""
    if original_df is None:
        return df.copy()
    out = df.copy()
    n = len(out)
    global_mean   = float(original_df[target_col].mean())
    global_median = float(original_df[target_col].median())
    for col in base_features:
        if col not in original_df.columns:
            continue
        stats = (
            original_df.groupby(col)[target_col]
            .agg(["mean", "median", "std", "skew", "count"])
            .reset_index()
        )
        stats.columns = [col] + [f"orig_{col}_{s}" for s in ["mean", "median", "std", "skew", "count"]]
        out = out.merge(stats, on=col, how="left")
        assert len(out) == n, f"Merge expanded rows for column {col}"
        out = out.fillna({
            f"orig_{col}_mean":   global_mean,
            f"orig_{col}_median": global_median,
            f"orig_{col}_std":    0.0,
            f"orig_{col}_skew":   0.0,
            f"orig_{col}_count":  0.0,
        })
    return out

# Ensembling helper
def rank01(x: np.ndarray) -> np.ndarray:
    """Map array to [0, 1] via average-rank normalisation (scipy rankdata)."""
    x = np.asarray(x, dtype=np.float64)
    r = rankdata(x, method="average")
    return (r - 0.5) / len(r)


## 4. Data Access (Kaggle / Colab / Local)


In [4]:
# Determine data directory
kaggle_dir = CFG['kaggle_comp_dir']
local_dir  = CFG['local_data_dir']

if kaggle_dir.exists():
    data_dir = kaggle_dir
    print("[Kaggle] Using mounted competition data:", data_dir)
else:
    data_dir = local_dir
    if _local_data_ready(data_dir):
        print("[Local] Data already present:", data_dir)
    else:
        print("[Download] Fetching via Kaggle API...")
        kaggle_json = _ensure_kaggle_json_colab(Path("/content/kaggle.json"))
        _install_kaggle_json(kaggle_json)
        _download_competition(data_dir)
        print("Download complete.")


[Kaggle] Using mounted competition data: /kaggle/input/competitions/playground-series-s6e2


## 5. Data Loading


In [5]:
# Load competition files
for fname in CFG['need_files']:
    if not (data_dir / fname).exists():
        raise FileNotFoundError(f"Required file missing: {data_dir / fname}")

train = pd.read_csv(data_dir / "train.csv")
test  = pd.read_csv(data_dir / "test.csv")
sub   = pd.read_csv(data_dir / "sample_submission.csv")

# Optional external dataset (Kaggle-only)
ext_path = CFG['kaggle_ext_path']
original = pd.read_csv(ext_path) if ext_path.exists() else None

print(f"train: {train.shape}, test: {test.shape}, sub: {sub.shape}")
print(f"external: {'loaded' if original is not None else 'not available'}")

# Minimal sanity check
check_data_quality(train, "Train")
check_data_quality(test,  "Test")


train: (630000, 15), test: (270000, 14), sub: (270000, 2)
external: not available
[Train] rows=630000, duplicates=0, total_NaN=0
[Test] rows=270000, duplicates=0, total_NaN=0


## 6. Preprocessing & Feature Engineering


In [6]:
# Basic columns
TARGET_COL = CFG['target_col']
ID_COL     = CFG['id_col']
META_COLS  = [TARGET_COL, ID_COL, "is_external"]

# 1) Encode target if needed
train_comp = train.copy()
if not pd.api.types.is_numeric_dtype(train_comp[TARGET_COL]):
    le = LabelEncoder()
    train_comp[TARGET_COL] = le.fit_transform(train_comp[TARGET_COL])
    if original is not None and TARGET_COL in original.columns:
        original[TARGET_COL] = le.transform(original[TARGET_COL])

# 2) Align and append external data
if original is not None:
    if TARGET_COL not in original.columns:
        raise ValueError("External dataset missing target column.")
    if not pd.api.types.is_numeric_dtype(original[TARGET_COL]):
        original[TARGET_COL] = encode_target_strict(original[TARGET_COL])

    orig_aligned = original.copy()
    for col in train_comp.columns:
        if col not in orig_aligned.columns:
            orig_aligned[col] = np.nan
    if ID_COL not in orig_aligned.columns:
        orig_aligned[ID_COL] = -(np.arange(len(orig_aligned)) + 1)
    orig_aligned = orig_aligned[train_comp.columns]

    train_full = pd.concat([train_comp, orig_aligned], ignore_index=True)
    train_full["is_external"] = [0] * len(train_comp) + [1] * len(orig_aligned)
else:
    train_full = train_comp.copy()
    train_full["is_external"] = 0

train = train_full
print(f"Combined train: {train.shape}")

# 3) Define categorical columns
BASE_FEATURES = [c for c in train.columns if c not in META_COLS]

# 4) Feature engineering: external target statistics
train_fe = add_external_target_stats(train, original, BASE_FEATURES, TARGET_COL)
test_fe  = add_external_target_stats(test,  original, BASE_FEATURES, TARGET_COL)

# 5) Build X / y / X_test
X = train_fe.drop(columns=[c for c in META_COLS if c in train_fe.columns])
y = train_fe[TARGET_COL].astype("int8")
X_test = test_fe.drop(columns=[c for c in [ID_COL] if c in test_fe.columns])

# All columns treated as categorical strings (matches RealMLP CV notebook approach)
for col in X.columns:
    X[col]      = X[col].astype(str).astype("category")
    X_test[col] = X_test[col].astype(str).astype("category")

cat_cols = list(X.columns)

# Sanity checks
assert len(X) == len(y), "X/y length mismatch"
assert len(X_test) == len(test), "X_test/test length mismatch"
print(f"X: {X.shape}  |  X_test: {X_test.shape}")


Combined train: (630000, 16)
X: (630000, 13)  |  X_test: (270000, 13)


## 7. Validation Setup


In [7]:
# CV configuration
N_FOLDS = CFG['n_folds']
RANDOM_STATE = CFG['random_state']

# Competition rows only (exclude external data from CV)
comp_mask = (train["is_external"] == 0).values

X_comp = X.loc[comp_mask].reset_index(drop=True)
y_comp = y.loc[comp_mask].reset_index(drop=True)
comp_ids = train.loc[comp_mask, ID_COL].reset_index(drop=True)

X_test_comp = X_test.copy()
test_ids    = test[ID_COL].reset_index(drop=True)

# Shared CV splitter
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# Sanity checks
assert len(X_comp) == len(y_comp), "X_comp/y_comp length mismatch"
assert len(test_ids) == len(X_test_comp), "test_ids/X_test_comp length mismatch"
print(f"X_comp: {X_comp.shape}  |  y_comp: {y_comp.shape}  |  X_test_comp: {X_test_comp.shape}")
print(f"CV: {N_FOLDS}-fold stratified, seed={RANDOM_STATE}")


X_comp: (630000, 13)  |  y_comp: (630000,)  |  X_test_comp: (270000, 13)
CV: 5-fold stratified, seed=42


## 8. CatBoost Training (OOF + Test)


In [8]:
# Train CatBoost base model
cat_features_idx = [X_comp.columns.get_loc(c) for c in cat_cols]

oof_cat  = np.zeros(len(X_comp), dtype="float32")
test_cat = np.zeros(len(X_test_comp), dtype="float32")

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_comp, y_comp), start=1):
    print(f"[CatBoost] Fold {fold}/{N_FOLDS}")
    X_tr, X_val = X_comp.iloc[tr_idx], X_comp.iloc[val_idx]
    y_tr, y_val = y_comp.iloc[tr_idx], y_comp.iloc[val_idx]

    train_pool = Pool(X_tr, label=y_tr, cat_features=cat_features_idx)
    valid_pool = Pool(X_val, label=y_val, cat_features=cat_features_idx)
    test_pool  = Pool(X_test_comp, cat_features=cat_features_idx)

    model = CatBoostClassifier(
        loss_function="Logloss", 
        eval_metric="AUC",
        learning_rate=0.03, 
        depth=8, 
        l2_leaf_reg=10,
        random_seed=RANDOM_STATE + fold,
        random_strength=1.0,
        bagging_temperature=0.5,
        border_count=128,
        iterations=4000, 
        od_type="Iter", 
        od_wait=200,
        verbose=200,
        task_type="GPU" if DEVICE == "cuda" else "CPU",
    )
    model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

    oof_cat[val_idx]  = model.predict_proba(valid_pool)[:, 1].astype("float32")
    test_cat         += (model.predict_proba(test_pool)[:, 1] / N_FOLDS).astype("float32")

assert len(oof_cat) == len(y_comp), "oof_cat length mismatch"
cat_oof_auc = roc_auc_score(y_comp, oof_cat)
print(f"\n[CatBoost Base] OOF AUC = {cat_oof_auc:.6f}")


[CatBoost] Fold 1/5


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9456944	best: 0.9456944 (0)	total: 263ms	remaining: 17m 30s
200:	test: 0.9558481	best: 0.9558481 (200)	total: 13.3s	remaining: 4m 12s
400:	test: 0.9559794	best: 0.9559794 (399)	total: 26.6s	remaining: 3m 58s
600:	test: 0.9560015	best: 0.9560024 (574)	total: 39.6s	remaining: 3m 43s
800:	test: 0.9560007	best: 0.9560046 (660)	total: 52.7s	remaining: 3m 30s
bestTest = 0.95600456
bestIteration = 660
Shrink model to first 661 iterations.
[CatBoost] Fold 2/5


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9445190	best: 0.9445190 (0)	total: 107ms	remaining: 7m 6s
200:	test: 0.9546772	best: 0.9546772 (200)	total: 13.4s	remaining: 4m 12s
400:	test: 0.9548260	best: 0.9548260 (399)	total: 26.6s	remaining: 3m 59s
600:	test: 0.9548488	best: 0.9548492 (590)	total: 39.6s	remaining: 3m 43s
800:	test: 0.9548565	best: 0.9548568 (746)	total: 52.8s	remaining: 3m 30s
bestTest = 0.954856813
bestIteration = 746
Shrink model to first 747 iterations.
[CatBoost] Fold 3/5


...

0:	test: 0.9450125	best: 0.9450125 (0)	total: 97.9ms	remaining: 6m 31s
200:	test: 0.9554693	best: 0.9554693 (200)	total: 13.2s	remaining: 4m 9s
400:	test: 0.9556178	best: 0.9556193 (391)	total: 26.4s	remaining: 3m 57s
600:	test: 0.9556359	best: 0.9556379 (577)	total: 39.3s	remaining: 3m 42s
800:	test: 0.9556339	best: 0.9556386 (640)	total: 52.4s	remaining: 3m 29s
bestTest = 0.9556386471
bestIteration = 640
Shrink model to first 641 iterations.
[CatBoost] Fold 4/5


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9443868	best: 0.9443868 (0)	total: 103ms	remaining: 6m 52s
200:	test: 0.9550557	best: 0.9550557 (200)	total: 13.2s	remaining: 4m 9s
400:	test: 0.9552122	best: 0.9552122 (400)	total: 26.5s	remaining: 3m 57s
600:	test: 0.9552305	best: 0.9552305 (600)	total: 39.3s	remaining: 3m 42s
800:	test: 0.9552352	best: 0.9552362 (752)	total: 52.3s	remaining: 3m 28s
1000:	test: 0.9552356	best: 0.9552395 (891)	total: 1m 5s	remaining: 3m 16s
bestTest = 0.9552394748
bestIteration = 891
Shrink model to first 892 iterations.
[CatBoost] Fold 5/5


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9466468	best: 0.9466468 (0)	total: 97.2ms	remaining: 6m 28s
200:	test: 0.9559561	best: 0.9559561 (200)	total: 13.2s	remaining: 4m 9s
400:	test: 0.9561105	best: 0.9561105 (400)	total: 26.4s	remaining: 3m 56s
600:	test: 0.9561224	best: 0.9561231 (579)	total: 39.4s	remaining: 3m 42s
800:	test: 0.9561270	best: 0.9561276 (780)	total: 52.1s	remaining: 3m 28s
1000:	test: 0.9561189	best: 0.9561288 (885)	total: 1m 5s	remaining: 3m 14s
bestTest = 0.9561288357
bestIteration = 885
Shrink model to first 886 iterations.

[CatBoost Base] OOF AUC = 0.955568


## 9. RealMLP Training (OOF + Test)


In [9]:
# RealMLP hyperparameters (keep base settings)
REALMLP_PARAMS = {
    'random_state': RANDOM_STATE,
    'verbosity': 2,
    'n_epochs': 100,
    'batch_size': 2**12,
    'n_ens': 8,
    'use_early_stopping': True,
    'early_stopping_additive_patience': 20,
    'early_stopping_multiplicative_patience': 1,
    'act': "mish",
    'embedding_size': 8,
    'first_layer_lr_factor': 0.5962121993798933,
    'val_metric_name': '1-auc_ovr',
    'hidden_sizes': "rectangular",
    'hidden_width': 384,
    'lr': 0.04,
    'ls_eps': 0.011498317194338772,
    'ls_eps_sched': "coslog4",
    'max_one_hot_cat_size': 18,
    'n_hidden_layers': 4,
    'p_drop': 0.07301419697186451,
    'p_drop_sched': "flat_cos",
    'plr_hidden_1': 16,
    'plr_hidden_2': 8,
    'plr_lr_factor': 0.1151437622270563,
    'plr_sigma': 2.3316811282666916,
    'scale_lr_factor': 2.244801835541429,
    'sq_mom': 1.0 - 0.011834054955582318,
    'wd': 0.02369230879235962,
    'device': DEVICE,
}

def make_realmlp(params: dict, cat_col_names: list[str]) -> RealMLP_TD_Classifier:
    """Instantiate RealMLP; falls back gracefully if cat_col_names is unsupported."""
    try:
        return RealMLP_TD_Classifier(**params, cat_col_names=cat_col_names)
    except TypeError:
        return RealMLP_TD_Classifier(**params)

oof_realmlp  = np.zeros(len(X_comp), dtype="float32")
test_realmlp = np.zeros(len(X_test_comp), dtype="float32")
fold_scores  = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_comp, y_comp), start=1):
    print(f"\n--- RealMLP Fold {fold}/{N_FOLDS} ---")
    X_tr, X_val = X_comp.iloc[train_idx], X_comp.iloc[val_idx]
    y_tr, y_val = y_comp.iloc[train_idx], y_comp.iloc[val_idx]

    model = make_realmlp(REALMLP_PARAMS, cat_cols)
    model.fit(X_tr, y_tr.values, X_val, y_val.values)

    val_probs  = model.predict_proba(X_val)[:, 1]
    test_probs = model.predict_proba(X_test_comp)[:, 1]

    oof_realmlp[val_idx]  = val_probs.astype("float32")
    test_realmlp         += (test_probs / N_FOLDS).astype("float32")

    score = roc_auc_score(y_val, val_probs)
    fold_scores.append(score)
    print(f"Fold {fold} AUC: {score:.6f}")

    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

assert len(oof_realmlp) == len(y_comp), "oof_realmlp length mismatch"
print(f"\n[RealMLP] OOF AUC = {roc_auc_score(y_comp, oof_realmlp):.6f}")
print("Fold AUCs:", [f"{s:.4f}" for s in fold_scores])



--- RealMLP Fold 1/5 ---
Columns classified as continuous: []
Columns classified as categorical: ['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/100: val 1-auc_ovr = 0.047396
Epoch 2/100: val 1-auc_ovr = 0.044425
Epoch 3/100: val 1-auc_ovr = 0.043934
Epoch 4/100: val 1-auc_ovr = 0.043966
Epoch 5/100: val 1-auc_ovr = 0.043908
Epoch 6/100: val 1-auc_ovr = 0.043883
Epoch 7/100: val 1-auc_ovr = 0.043881
Epoch 8/100: val 1-auc_ovr = 0.043909
Epoch 9/100: val 1-auc_ovr = 0.043943
Epoch 10/100: val 1-auc_ovr = 0.043965
Epoch 11/100: val 1-auc_ovr = 0.043936
Epoch 12/100: val 1-auc_ovr = 0.043983
Epoch 13/100: val 1-auc_ovr = 0.043933
Epoch 14/100: val 1-auc_ovr = 0.043955
Epoch 15/100: val 1-auc_ovr = 0.043933
Epoch 16/100: val 1-auc_ovr = 0.043922
Epoch 17/100: val 1-auc_ovr = 0.043947
Epoch 18/100: val 1-auc_ovr = 0.043961
Epoch 19/100: val 1-auc_ovr = 0.043973
Epoch 20/100: val 1-auc_ovr = 0.043979
Epoch 21/100: val 1-auc_ovr = 0.043983
Epoch 22/100: val 1-auc_ovr = 0.043984
Epoch 23/100: val 1-auc_ovr = 0.044007
Epoch 24/100: val 1-auc_ovr = 0.044003
Epoch 25/100: val 1-auc_ovr = 0.043981
Epoch 26/100: val 1-auc_ovr = 0.04

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Fold 1 AUC: 0.956119

--- RealMLP Fold 2/5 ---
Columns classified as continuous: []
Columns classified as categorical: ['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']


...

Fold 4 AUC: 0.955436

--- RealMLP Fold 5/5 ---
Columns classified as continuous: []
Columns classified as categorical: ['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/100: val 1-auc_ovr = 0.047628
Epoch 2/100: val 1-auc_ovr = 0.044322
Epoch 3/100: val 1-auc_ovr = 0.043898
Epoch 4/100: val 1-auc_ovr = 0.043818
Epoch 5/100: val 1-auc_ovr = 0.043801
Epoch 6/100: val 1-auc_ovr = 0.043785
Epoch 7/100: val 1-auc_ovr = 0.043774
Epoch 8/100: val 1-auc_ovr = 0.043792
Epoch 9/100: val 1-auc_ovr = 0.043812
Epoch 10/100: val 1-auc_ovr = 0.043804
Epoch 11/100: val 1-auc_ovr = 0.043865
Epoch 12/100: val 1-auc_ovr = 0.043801
Epoch 13/100: val 1-auc_ovr = 0.043811
Epoch 14/100: val 1-auc_ovr = 0.043784
Epoch 15/100: val 1-auc_ovr = 0.043829
Epoch 16/100: val 1-auc_ovr = 0.043795
Epoch 17/100: val 1-auc_ovr = 0.043799
Epoch 18/100: val 1-auc_ovr = 0.043831
Epoch 19/100: val 1-auc_ovr = 0.043840
Epoch 20/100: val 1-auc_ovr = 0.043843
Epoch 21/100: val 1-auc_ovr = 0.043851
Epoch 22/100: val 1-auc_ovr = 0.043857
Epoch 23/100: val 1-auc_ovr = 0.043858
Epoch 24/100: val 1-auc_ovr = 0.043841
Epoch 25/100: val 1-auc_ovr = 0.043835
Epoch 26/100: val 1-auc_ovr = 0.04

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Fold 5 AUC: 0.956226

[RealMLP] OOF AUC = 0.955657
Fold AUCs: ['0.9561', '0.9549', '0.9558', '0.9554', '0.9562']


## 10. OOF Evaluation & Rank Mean


In [10]:
# Rank-mean blend (2 models)
rank_mean_2models = np.mean(
    [rank01(oof_cat), rank01(oof_realmlp)],
    axis=0,
)
rank_mean_2models_test = np.mean(
    [rank01(test_cat), rank01(test_realmlp)],
    axis=0,
)

# AUC comparisons
print("=== OOF AUC (base vs rank mean) ===")
print(f"  CatBoost only      : {roc_auc_score(y_comp, oof_cat):.6f}")
print(f"  RealMLP only       : {roc_auc_score(y_comp, oof_realmlp):.6f}")
print(f"  Rank mean (2 models): {roc_auc_score(y_comp, rank_mean_2models):.6f}")


=== OOF AUC (base vs rank mean) ===
  CatBoost only      : 0.955568
  RealMLP only       : 0.955657
  Rank mean (2 models): 0.955708


## 11. Submission Candidates


In [11]:
# Candidate dicts
submission_candidates = {
    "cat_only":          test_cat.copy(),
    "rank_mean_2models": rank_mean_2models_test.copy(),
}

oof_candidates = {
    "cat_only":          oof_cat.copy(),
    "rank_mean_2models": rank_mean_2models.copy(),
}

selected_submission_name = "rank_mean_2models"
assert selected_submission_name in submission_candidates, "selected_submission_name missing"
print("Selected candidate:", selected_submission_name)


Selected candidate: rank_mean_2models


## 12. Save CSVs


In [12]:
# Save candidate submissions
for name, pred in submission_candidates.items():
    sub_df = pd.DataFrame({
        "id": test_ids.values if hasattr(test_ids, "values") else test_ids,
        "Heart Disease": np.clip(pred.astype(np.float64), 0.0, 1.0),
    })
    fname = f"submission_{name}.csv"
    sub_df.to_csv(fname, index=False)
    print(f"Saved: {fname}  shape={sub_df.shape}")

# Final submission
final_pred = np.clip(
    submission_candidates[selected_submission_name].astype(np.float64),
    0.0, 1.0,
)
submission = pd.DataFrame({
    "id": test_ids.values if hasattr(test_ids, "values") else test_ids,
    "Heart Disease": final_pred,
})
submission.to_csv("submission.csv", index=False)
print(f"Saved final: submission.csv  ({selected_submission_name})")


Saved: submission_cat_only.csv  shape=(270000, 2)
Saved: submission_rank_mean_2models.csv  shape=(270000, 2)
Saved final: submission.csv  (rank_mean_2models)


## 13. Final Summary


In [13]:
# Final OOF summary
print("OOF AUC summary:")
for name in ["cat_only", "rank_mean_2models"]:
    auc = roc_auc_score(y_comp, oof_candidates[name])
    marker = " ← selected" if name == selected_submission_name else ""
    print(f"  [{name}] {auc:.6f}{marker}")

print("Saved files:")
print("  submission_cat_only.csv")
print("  submission_rank_mean_2models.csv")
print("  submission.csv")


OOF AUC summary:
  [cat_only] 0.955568
  [rank_mean_2models] 0.955708 ← selected
Saved files:
  submission_cat_only.csv
  submission_rank_mean_2models.csv
  submission.csv
